# T06 — Deteksi Perubahan Lahan Tanpa Label di Kawasan PSN Merauke: Optik, Radar, Embedding Foundation Model, dan Ensembel

Tidak ada data latih. Setiap metode menghasilkan **skor perubahan** 2023 → 2025 dan ambangnya ditentukan otomatis (Otsu). Data acuan konsensus (Hansen + RADD/TMF) **hanya dipakai untuk menilai**, tidak untuk melatih.

| Kode | Metode | Data |
|---|---|---|
| M1 | Change Vector Analysis (CVA) | Sentinel-2, 6 band |
| M2 | Selisih NDVI (dNDVI) | Sentinel-2 |
| M3 | PCA + k-means (2 klaster) | Sentinel-2, selisih band + rata-rata 3×3 |
| M4 | Isolation Forest (deteksi anomali) | Sentinel-2, fitur yang sama dengan M3 |
| M5 | Log-ratio radar | Sentinel-1 VV, VH |
| M6 | Jarak kosinus embedding | AlphaEarth 2023 vs 2025 |
| M7 | Ensembel skor ternormalisasi (M1 + M5 + M6) | Gabungan |

| Tahap | Waktu |
|---|---|
| 0–2 Instalasi, login, Blok E dan F | 3 menit |
| 3 Skor perubahan M1, M2, M5, M6 + fitur M3/M4 | 1 menit |
| 4 Label acuan, sampel berlabel (penilaian) dan sampel tanpa label (ambang, M3, M4) | 10–15 menit |
| 5 Ambang Otsu, M3, M4, M7, metrik (F1, AUC, McNemar) | 2 menit |
| 6 Peta Blok E dan gambar dua bahasa | 5 menit |

*Runtime → Jalankan semua*. Hasil: `T06_hasil.zip`.

In [ ]:
!pip install -q earthengine-api scikit-learn pandas matplotlib seaborn scipy openpyxl requests

In [ ]:
import ee
GEE_PROJECT = 'riset-psn-pps'
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate(force=True); ee.Initialize(project=GEE_PROJECT)
print('Earth Engine siap:', ee.String('OK').getInfo())

In [ ]:
import os, json, time, zipfile, requests
import numpy as np, pandas as pd
TOPIC = 'T06'; SEED = 42
OUT = f'/content/{TOPIC}_hasil' if os.path.exists('/content') else f'./{TOPIC}_hasil'
DIR_DATA, DIR_TAB, DIR_FIG = [os.path.join(OUT, d) for d in ('07_dataset', '03_tabel', '02_gambar')]
for d in (DIR_DATA, DIR_TAB, DIR_FIG): os.makedirs(d, exist_ok=True)
CRS = 'EPSG:32754'; BLOCK_KM = 20
PRE, POST = ('2023-06-01', '2023-11-01'), ('2025-06-01', '2025-11-01')       # musim kemarau, sebelum dan sesudah
N_LAB = 1500                     # titik berlabel per kelas per blok (hanya untuk penilaian)
N_UNL = 15000                    # piksel tanpa label per blok (ambang Otsu, M3, M4)
USED = {'A': (140.546, -8.232), 'B': (140.7276, -8.2324), 'C': (140.7278, -7.8706), 'D': (139.2781, -7.5055)}   # blok topik lain
M_NAME = {'M1': 'CVA (S2)', 'M2': 'dNDVI (S2)', 'M3': 'PCA + k-means (S2)', 'M4': 'Isolation Forest (S2)', 'M5': 'Log-ratio (S1)', 'M6': 'Embedding AlphaEarth', 'M7': 'Ensembel'}
M_EN = {'M1': 'CVA (S2)', 'M2': 'dNDVI (S2)', 'M3': 'PCA + k-means (S2)', 'M4': 'Isolation Forest (S2)', 'M5': 'Log-ratio (S1)', 'M6': 'AlphaEarth embedding', 'M7': 'Ensemble'}
COLS_EN = {'metode': 'method', 'blok': 'block', 'presisi': 'precision', 'recall': 'recall', 'F1': 'F1', 'OA': 'OA', 'kappa': 'kappa', 'AUC': 'AUC',
           'F1_ambang_terbaik': 'F1_best_threshold', 'cakupan_persen': 'coverage_pct', 'ambang': 'threshold', 'pasangan': 'pair', 'p_nilai': 'p_value',
           'signifikan': 'significant', 'luas_perubahan_ha': 'change_area_ha', 'luas_hansen_ha': 'hansen_loss_ha', 'peringkat': 'rank', 'loss_2024_2025_ha': 'loss_2024_2025_ha',
           'lon': 'lon', 'lat': 'lat', 'kelas': 'class', 'jumlah': 'count', 'keterangan': 'note'}
VAL_EN = {**{v: M_EN[k] for k, v in M_NAME.items()}, 'ya': 'yes', 'tidak': 'no', 'gabungan': 'pooled', 'berubah': 'change', 'tidak berubah': 'no change'}
def save_tab(df, name_id, name_en):
    d = df.copy()
    for c in d.columns:
        if not pd.api.types.is_numeric_dtype(d[c]): d[c] = d[c].map(lambda v: VAL_EN.get(v, v))
    df.to_csv(os.path.join(DIR_TAB, f'T06_tabel_{name_id}_ID.csv'), index=False)
    d.rename(columns=COLS_EN).to_csv(os.path.join(DIR_TAB, f'T06_table_{name_en}_EN.csv'), index=False)
np.random.seed(SEED); print('Folder keluaran:', OUT)

## Tahap 2 — Blok E dan F
Grid 20 × 20 km yang sama; sel diurutkan menurut kehilangan tutupan pohon Hansen 2024–2025 (resolusi 30 m). Sel yang sudah dipakai T01, T02, dan T05 (Blok A–D) dilewati, lalu dua sel berikutnya menjadi **Blok E** dan **Blok F**.

In [ ]:
kab = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM2_NAME', 'Merauke')); kab_geom = kab.geometry()
gfc = ee.Image('UMD/hansen/global_forest_change_2025_v1_13'); lossyear = gfc.select('lossyear').unmask(0); tc2000 = gfc.select('treecover2000')
loss_ha = ee.Image.pixelArea().divide(1e4).updateMask(lossyear.gte(24).And(lossyear.lte(25))).rename('loss_ha')
grid = kab_geom.coveringGrid(ee.Projection(CRS).atScale(BLOCK_KM * 1000))
top = (loss_ha.reduceRegions(collection=grid, reducer=ee.Reducer.sum(), scale=30, tileScale=8).filter(ee.Filter.gt('sum', 0)).sort('sum', False)
       .limit(10).map(lambda f: f.set('cx', f.geometry().centroid(1).coordinates().get(0), 'cy', f.geometry().centroid(1).coordinates().get(1)))).getInfo()['features']
rows, BLK = [], {}
for i, f in enumerate(top):
    cx, cy = f['properties']['cx'], f['properties']['cy']
    used = [k for k, (ux, uy) in USED.items() if abs(cx - ux) < 0.05 and abs(cy - uy) < 0.05]
    rows.append({'peringkat': i + 1, 'lon': round(cx, 4), 'lat': round(cy, 4), 'loss_2024_2025_ha': round(f['properties']['sum'], 1), 'keterangan': f'dipakai Blok {used[0]}' if used else ''})
    if not used and len(BLK) < 2:
        b = 'EF'[len(BLK)]; BLK[b] = ee.Geometry(f['geometry']); rows[-1]['keterangan'] = f'Blok {b} (T06)'
blocks = pd.DataFrame(rows); print(blocks.to_string(index=False)); save_tab(blocks, 'kandidat_blok', 'candidate_blocks')
for b, g in BLK.items():
    json.dump({'type': 'FeatureCollection', 'features': [{'type': 'Feature', 'properties': {'blok': b}, 'geometry': g.transform('EPSG:4326', 1).getInfo()}]}, open(os.path.join(DIR_DATA, f'T06_blok{b}.geojson'), 'w'))
REG = BLK['E'].union(BLK['F'], 1).buffer(500)

## Tahap 3 — Skor perubahan (tanpa label)
* **M1 CVA:** panjang vektor selisih 6 band Sentinel-2 (B2, B3, B4, B8, B11, B12) 2025 − 2023.
* **M2 dNDVI:** NDVI 2023 − NDVI 2025 (positif = vegetasi hilang).
* **M5 Sentinel-1:** panjang vektor selisih dB (VV, VH) median 2025 − 2023.
* **M6 AlphaEarth:** 1 − kemiripan kosinus embedding 2023 dan 2025.
* Fitur M3/M4: 6 selisih band + rata-rata 3 × 3 masing-masing (12 fitur).

In [ ]:
B6 = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
cs_plus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
def s2(a, b):
    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(REG).filterDate(a, b).linkCollection(cs_plus, ['cs_cdf'])
            .map(lambda i: i.updateMask(i.select('cs_cdf').gte(0.6)).select(B6).divide(10000)).median())
def s1(a, b):
    return ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(REG).filterDate(a, b).filter(ee.Filter.eq('instrumentMode', 'IW')) \
             .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')).select(['VV', 'VH']).median()
pre, post = s2(*PRE), s2(*POST)
diff = post.subtract(pre).rename([f'd{b}' for b in B6])
m1 = diff.pow(2).reduce(ee.Reducer.sum()).sqrt().rename('M1')
m2 = pre.normalizedDifference(['B8', 'B4']).subtract(post.normalizedDifference(['B8', 'B4'])).rename('M2')
ds1 = s1(*POST).subtract(s1(*PRE)); m5 = ds1.pow(2).reduce(ee.Reducer.sum()).sqrt().rename('M5')
AEB = [f'A{i:02d}' for i in range(64)]; AEC = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
ae = lambda y: AEC.filterDate(f'{y}-01-01', f'{y + 1}-01-01').filterBounds(REG).mosaic().select(AEB)
m6 = ee.Image(1).subtract(ae(2023).multiply(ae(2025)).reduce(ee.Reducer.sum())).rename('M6')
dmean = diff.focalMean(1, 'square').rename([f'm{b}' for b in B6])
FEAT34 = [f'd{b}' for b in B6] + [f'm{b}' for b in B6]
SCORES = ee.Image.cat([m1, m2, m5, m6, diff, dmean]).toFloat().unmask(-9999)
print('Skor dan fitur siap:', SCORES.bandNames().getInfo())

## Tahap 4 — Data acuan (hanya untuk penilaian) dan sampel
| Kelas acuan | Aturan |
|---|---|
| 1 Berubah (dibuka) | Hansen kehilangan 2024 atau 2025, dikonfirmasi RADD 2024–2025 atau JRC TMF Des-2025 = terdeforestasi/terdegradasi |
| 0 Tidak berubah | (tutupan pohon 2000 ≥ 50% atau < 10%) tanpa kehilangan s.d. 2025, tanpa RADD, bukan air (GSW < 50%) |

In [ ]:
radd = (ee.ImageCollection('projects/radar-wur/raddalert/v1').filterMetadata('layer', 'contains', 'alert')
        .filterMetadata('geography', 'equals', 'asia').sort('system:time_end', False).first())
rdate = radd.select('Date').unmask(0); tmf25 = ee.ImageCollection('projects/JRC/TMF/v1_2025/AnnualChanges').mosaic().select('Dec2025')
gsw = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').unmask(0)
chg = lossyear.gte(24).And(lossyear.lte(25)).And(rdate.gte(24001).And(rdate.lte(25366)).Or(tmf25.eq(2)).Or(tmf25.eq(3)))
stab = lossyear.eq(0).And(rdate.eq(0)).And(tc2000.gte(50).Or(tc2000.lt(10))).And(gsw.lt(50))
REF = ee.Image(-1).where(stab, 0).where(chg, 1).updateMask(gsw.lt(50)).rename('ref').toInt()
REF = REF.updateMask(REF.gte(0))
def fc_to_df(fc, page=1000):
    n = fc.size().getInfo(); lst = fc.toList(n); rows = []
    for s in range(0, n, page):
        for f in ee.FeatureCollection(lst.slice(s, s + page)).getInfo()['features']:
            p = dict(f['properties']); p['lon'], p['lat'] = f['geometry']['coordinates']; rows.append(p)
    return pd.DataFrame(rows)
F_LAB, F_UNL = os.path.join(DIR_DATA, 'T06_sampel_berlabel.csv'), os.path.join(DIR_DATA, 'T06_sampel_tanpa_label.csv')
HANSEN = {}
for b, g in BLK.items():
    HANSEN[b] = loss_ha.reduceRegion(ee.Reducer.sum(), g, 30, maxPixels=1e10, tileScale=8).get('loss_ha').getInfo()
if os.path.exists(F_LAB) and os.path.exists(F_UNL):
    LAB, UNL = pd.read_csv(F_LAB), pd.read_csv(F_UNL); print('Sampel dimuat ulang')
else:
    lab, unl = [], []
    for b, g in BLK.items():
        t0 = time.perf_counter()
        fc = REF.addBands(SCORES).stratifiedSample(numPoints=N_LAB, classBand='ref', region=g, scale=10, seed=SEED, geometries=True, tileScale=8, dropNulls=False)
        d = fc_to_df(fc); d['blok'] = b; lab.append(d)
        for k in range(N_UNL // 5000):                    # batas 5.000 baris per permintaan
            fc = SCORES.sample(region=g, scale=10, numPixels=5000, seed=SEED + k, geometries=False, tileScale=8)
            u = pd.DataFrame([f['properties'] for f in fc.getInfo()['features']]); u['blok'] = b; unl.append(u)
        print(f'Blok {b}: {len(d)} titik berlabel, {sum(len(x) for x in unl if x.blok.iloc[0] == b)} piksel tanpa label ({time.perf_counter() - t0:.0f} dtk)')
    LAB, UNL = pd.concat(lab, ignore_index=True), pd.concat(unl, ignore_index=True)
    LAB.to_csv(F_LAB, index=False); UNL.to_csv(F_UNL, index=False)
for D_ in (LAB, UNL): D_.replace(-9999, np.nan, inplace=True)
print(LAB.groupby(['blok', 'ref']).size().unstack())

## Tahap 5 — Ambang Otsu, M3, M4, M7, dan penilaian
* Ambang Otsu, PCA + k-means (M3), dan Isolation Forest (M4) dihitung **hanya dari sampel tanpa label** per blok.
* M7: setiap skor M1, M5, M6 dinormalisasi ke persentil 1–99 sampel tanpa label (0–1), dirata-rata, lalu Otsu.
* Metrik pada sampel berlabel: presisi, recall, F1 kelas berubah, OA, kappa; AUC (tanpa ambang) dan F1 dengan ambang terbaik (batas atas).

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, cohen_kappa_score, roc_auc_score, precision_recall_curve, roc_curve
from scipy.stats import binomtest
def otsu(v):
    v = np.asarray(v, float); v = v[np.isfinite(v)]; lo, hi = np.percentile(v, [0.5, 99.5]); v = np.clip(v, lo, hi)
    h, e = np.histogram(v, 256); c = (e[:-1] + e[1:]) / 2; w = np.cumsum(h); m = np.cumsum(h * c); wt, mt = w[-1], m[-1]
    b = (mt * w - m * wt) ** 2 / np.maximum(w * (wt - w), 1e-9); return float(c[np.nanargmax(b)])
SC, THR, rows, PRED = {}, [], [], {}
for b in BLK:
    L, U = LAB[LAB.blok == b].copy(), UNL[UNL.blok == b].copy()
    for m in ('M1', 'M2', 'M5', 'M6'):
        t = otsu(U[m]); THR.append({'blok': b, 'metode': M_NAME[m], 'ambang': round(t, 4)}); L[f'{m}_s'] = L[m]; L[f'{m}_p'] = (L[m] > t).astype(float).where(L[m].notna())
    okU, okL = U[FEAT34].notna().all(1), L[FEAT34].notna().all(1)
    sc = StandardScaler().fit(U.loc[okU, FEAT34]); XU, XL = sc.transform(U.loc[okU, FEAT34]), sc.transform(L.loc[okL, FEAT34])
    pca = PCA(n_components=3, random_state=SEED).fit(XU); km = KMeans(2, n_init=10, random_state=SEED).fit(pca.transform(XU))
    chg_c = int(np.argmax([np.linalg.norm(c) for c in km.cluster_centers_]))          # klaster yang lebih jauh dari "tanpa perubahan"
    dist = np.linalg.norm(pca.transform(XL) - km.cluster_centers_[1 - chg_c], axis=1) - np.linalg.norm(pca.transform(XL) - km.cluster_centers_[chg_c], axis=1)
    L.loc[okL, 'M3_s'] = dist; L.loc[okL, 'M3_p'] = (km.predict(pca.transform(XL)) == chg_c).astype(float)
    iso = IsolationForest(n_estimators=300, random_state=SEED).fit(XU); su = -iso.score_samples(XU); t4 = otsu(su)
    L.loc[okL, 'M4_s'] = -iso.score_samples(XL); L.loc[okL, 'M4_p'] = (L.loc[okL, 'M4_s'] > t4).astype(float)
    THR.append({'blok': b, 'metode': M_NAME['M4'], 'ambang': round(t4, 4)})
    norm = {}
    for m in ('M1', 'M5', 'M6'):
        lo, hi = np.nanpercentile(U[m], [1, 99]); norm[m] = (lo, hi)
    nz = lambda D_, m: ((D_[m] - norm[m][0]) / (norm[m][1] - norm[m][0])).clip(0, 1)
    eU = pd.concat([nz(U, m) for m in ('M1', 'M5', 'M6')], axis=1).mean(1, skipna=True); t7 = otsu(eU)
    L['M7_s'] = pd.concat([nz(L, m) for m in ('M1', 'M5', 'M6')], axis=1).mean(1, skipna=True); L['M7_p'] = (L['M7_s'] > t7).astype(float).where(L['M7_s'].notna())
    THR.append({'blok': b, 'metode': M_NAME['M7'], 'ambang': round(t7, 4)}); SC[b] = (L, norm, t7)
    for m in M_NAME:
        ok = L[f'{m}_p'].notna(); y, p, s = L.ref[ok].values, L[f'{m}_p'][ok].values.astype(int), L[f'{m}_s'][ok].values
        pr, rc, th = precision_recall_curve(y, s); fbest = np.nanmax(2 * pr * rc / np.maximum(pr + rc, 1e-9))
        rows.append({'blok': b, 'metode': M_NAME[m], 'presisi': precision_score(y, p, zero_division=0), 'recall': recall_score(y, p), 'F1': f1_score(y, p),
                     'OA': accuracy_score(y, p), 'kappa': cohen_kappa_score(y, p), 'AUC': roc_auc_score(y, s), 'F1_ambang_terbaik': fbest, 'cakupan_persen': 100 * ok.mean()})
        PRED[(b, m)] = L[f'{m}_p']
MET = pd.DataFrame(rows).round(4); save_tab(MET, 'metrik_per_blok', 'metrics_per_block'); save_tab(pd.DataFrame(THR), 'ambang_otsu', 'otsu_thresholds')
POOL = MET.groupby('metode', sort=False)[['presisi', 'recall', 'F1', 'OA', 'kappa', 'AUC', 'F1_ambang_terbaik', 'cakupan_persen']].mean().round(4).reset_index()
save_tab(POOL, 'metrik_rata2_blok', 'metrics_block_mean'); print(POOL.to_string(index=False))
# McNemar: metode terbaik vs lainnya (titik yang dinilai oleh keduanya)
best = POOL.sort_values('F1', ascending=False).metode.iloc[0]; bk = [k for k, v in M_NAME.items() if v == best][0]; mc = []
for b in BLK:
    L = SC[b][0]
    for m in M_NAME:
        if m == bk: continue
        ok = L[f'{m}_p'].notna() & L[f'{bk}_p'].notna(); y = L.ref[ok].values; a = L[f'{bk}_p'][ok].values == y; c = L[f'{m}_p'][ok].values == y
        n01, n10 = int((a & ~c).sum()), int((~a & c).sum()); p = binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
        mc.append({'blok': b, 'pasangan': f'{best} vs {M_NAME[m]}', 'n01': n01, 'n10': n10, 'p_nilai': round(p, 5), 'signifikan': 'ya' if p < 0.05 else 'tidak'})
MC = pd.DataFrame(mc); save_tab(MC, 'mcnemar', 'mcnemar'); print(MC.to_string(index=False))
pd.concat([SC[b][0] for b in BLK]).round(5).to_csv(os.path.join(DIR_DATA, 'T06_skor_dan_prediksi.csv'), index=False)

## Tahap 6 — Peta Blok E dan gambar dua bahasa

In [ ]:
def thumb(img, fn, g):
    url = img.getThumbURL({'region': g, 'dimensions': 700, 'format': 'png', 'crs': CRS}); open(fn, 'wb').write(requests.get(url, timeout=300).content); return fn
TH = pd.DataFrame(THR); gE = BLK['E']; Lr, norm, t7 = SC['E']
tE = {r.metode: r.ambang for r in TH[TH.blok == 'E'].itertuples()}
ens = ee.Image.cat([SCORES.select(m).updateMask(SCORES.select(m).neq(-9999)).subtract(norm[m][0]).divide(norm[m][1] - norm[m][0]).clamp(0, 1) for m in ('M1', 'M5', 'M6')]).reduce(ee.Reducer.mean())
MAPS = {'pre': thumb(pre.select(['B4', 'B3', 'B2']).visualize(min=0, max=0.15), os.path.join(DIR_FIG, 'T06_E_s2_2023.png'), gE),
        'post': thumb(post.select(['B4', 'B3', 'B2']).visualize(min=0, max=0.15), os.path.join(DIR_FIG, 'T06_E_s2_2025.png'), gE),
        'ref': thumb(REF.visualize(min=0, max=1, palette=['d9d9d9', 'd7301f']), os.path.join(DIR_FIG, 'T06_E_acuan.png'), gE)}
for m in ('M1', 'M5', 'M6'):
    MAPS[m] = thumb(SCORES.select(m).gt(tE[M_NAME[m]]).And(SCORES.select(m).neq(-9999)).selfMask().visualize(palette=['d7301f']), os.path.join(DIR_FIG, f'T06_E_{m}.png'), gE)
MAPS['M7'] = thumb(ens.gt(t7).selfMask().visualize(palette=['d7301f']), os.path.join(DIR_FIG, 'T06_E_M7.png'), gE)
AREA = []
for m, img in (('M1', SCORES.select('M1').gt(tE[M_NAME['M1']]).And(SCORES.select('M1').neq(-9999))), ('M5', SCORES.select('M5').gt(tE[M_NAME['M5']])),
               ('M6', SCORES.select('M6').gt(tE[M_NAME['M6']])), ('M7', ens.gt(t7))):
    a = ee.Image.pixelArea().divide(1e4).updateMask(img).reduceRegion(ee.Reducer.sum(), gE, 30, maxPixels=1e10, tileScale=8).values().get(0).getInfo()
    AREA.append({'blok': 'E', 'metode': M_NAME[m], 'luas_perubahan_ha': round(a or 0, 1), 'luas_hansen_ha': round(HANSEN['E'], 1)})
AREA = pd.DataFrame(AREA); save_tab(AREA, 'luas_perubahan_blokE', 'change_area_blockE'); print(AREA.to_string(index=False))

In [ ]:
# Gambar dua bahasa
import matplotlib.pyplot as plt
from PIL import Image
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10})
ORD = list(M_NAME)
L = {'ID': dict(f1='F1 kelas berubah (ambang Otsu)', auc='AUC (tanpa ambang)', fpr='Laju positif palsu', tpr='Laju positif benar', blk='Blok', nm=M_NAME,
                ttl=['Sentinel-2 2023', 'Sentinel-2 2025', 'Acuan (merah = dibuka)', 'M1 CVA', 'M5 Sentinel-1', 'M6 AlphaEarth', 'M7 Ensembel'], sc='Skor ternormalisasi', ch='Berubah', nc='Tidak berubah'),
     'EN': dict(f1='Change-class F1 (Otsu threshold)', auc='AUC (threshold-free)', fpr='False positive rate', tpr='True positive rate', blk='Block', nm=M_EN,
                ttl=['Sentinel-2 2023', 'Sentinel-2 2025', 'Reference (red = cleared)', 'M1 CVA', 'M5 Sentinel-1', 'M6 AlphaEarth', 'M7 Ensemble'], sc='Normalised score', ch='Change', nc='No change')}
COL = dict(zip(ORD, ['#4C78A8', '#9ecae9', '#72B7B2', '#54A24B', '#F58518', '#E45756', '#333333']))
for lang, t in L.items():
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    for ax, met, lab in zip(axs, ['F1', 'AUC'], [t['f1'], t['auc']]):
        piv = MET.pivot(index='metode', columns='blok', values=met).reindex([M_NAME[k] for k in ORD]); x = np.arange(len(ORD))
        for i, b in enumerate(BLK): ax.bar(x + (i - 0.5) * 0.38, piv[b], 0.38, label=f"{t['blk']} {b}", color=['#4C78A8', '#F58518'][i])
        ax.set_xticks(x); ax.set_xticklabels([t['nm'][k] for k in ORD], rotation=30, ha='right', fontsize=8); ax.set_ylabel(lab); ax.set_ylim(0, 1); ax.grid(axis='y', alpha=0.3)
    axs[0].legend(fontsize=8); fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'T06_gambar_F1_AUC_{lang}.png'), dpi=300); plt.close(fig)
    fig, axs = plt.subplots(1, len(BLK), figsize=(11, 4.4))
    for ax, b in zip(axs, BLK):
        Lb = SC[b][0]
        for m in ORD:
            ok = Lb[f'{m}_s'].notna(); fpr, tpr, _ = roc_curve(Lb.ref[ok], Lb[f'{m}_s'][ok]); ax.plot(fpr, tpr, color=COL[m], lw=1.4, label=t['nm'][m])
        ax.plot([0, 1], [0, 1], 'k:', lw=0.7); ax.set_title(f"{t['blk']} {b}"); ax.set_xlabel(t['fpr']); ax.set_ylabel(t['tpr'])
    axs[-1].legend(fontsize=7, loc='lower right'); fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'T06_gambar_ROC_{lang}.png'), dpi=300); plt.close(fig)
    fig, axs = plt.subplots(1, 3, figsize=(12, 3.4)); Lb = SC['E'][0]
    for ax, m in zip(axs, ('M1', 'M5', 'M6')):
        for r, c, lab in ((1, '#d7301f', t['ch']), (0, '#7f7f7f', t['nc'])):
            v = Lb[Lb.ref == r][m].dropna(); ax.hist(v, 60, color=c, alpha=0.55, density=True, label=lab)
        ax.axvline(tE[M_NAME[m]], color='k', ls='--', lw=1); ax.set_title(t['nm'][m], fontsize=10)
    axs[0].legend(fontsize=8); fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'T06_gambar_distribusi_skor_{lang}.png'), dpi=300); plt.close(fig)
    keys = ['pre', 'post', 'ref', 'M1', 'M5', 'M6', 'M7']; fig, axs = plt.subplots(2, 4, figsize=(14, 7.4))
    for ax in axs.ravel(): ax.axis('off')
    for ax, k, ttl in zip(axs.ravel(), keys, t['ttl']):
        im = Image.open(MAPS[k]).convert('RGBA')
        if k in ('M1', 'M5', 'M6', 'M7'):
            bg = Image.new('RGBA', im.size, (235, 235, 235, 255)); bg.alpha_composite(im); im = bg
        ax.imshow(im); ax.set_title(ttl, fontsize=10)
    fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'T06_gambar_peta_blokE_{lang}.png'), dpi=220); plt.close(fig)
print(sorted(os.listdir(DIR_FIG)))

In [ ]:
summary = {'blok': blocks.to_dict('records'), 'hansen_loss_ha': HANSEN, 'n_berlabel': {f'{k[0]}_{k[1]}': int(v) for k, v in LAB.groupby(['blok', 'ref']).size().items()},
           'metrik_rata2': POOL.to_dict('records'), 'metrik_per_blok': MET.to_dict('records'), 'mcnemar': MC.to_dict('records'), 'luas_blokE': AREA.to_dict('records'), 'ambang': THR}
json.dump(summary, open(os.path.join(DIR_TAB, 'T06_hasil_ringkas.json'), 'w'), indent=2, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
zpath = '/content/T06_hasil.zip' if os.path.exists('/content') else 'T06_hasil.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, fs_ in os.walk(OUT):
        for fn in fs_: z.write(os.path.join(root, fn), os.path.relpath(os.path.join(root, fn), os.path.dirname(OUT)))
print(POOL.to_string(index=False))
try:
    from google.colab import files; files.download(zpath)
except Exception:
    print('Arsip:', zpath)

Selesai. Upload `T06_hasil.zip` ke Claude.